<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset MNIST handwritten digits utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (MNIST handwritten digits)

- Utilize os arquivos `*.csv` disponibilizados via Google Classroom
- Use: `as_frame=False`
- Use: `mnist_784`


## Importações

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas com sucesso!")


# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `MNIST handwritten digits` (arquivos de treino e teste)
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`


Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
def load_data(seed=42):
    """
    Carrega o dataset MNIST a partir dos arquivos CSV,
    separa treino/validação e retorna os conjuntos prontos para uso.

    Parâmetros:
        seed (int): semente para reprodutibilidade do split

    Retorna:
        X_train, X_val, y_train, y_val (arrays numpy)
    """
    # Carregando os arquivos CSV disponibilizados pelo professor
    train_df = pd.read_csv('mnist_train.csv')
    test_df  = pd.read_csv('mnist_test.csv')

    # Separando features (pixels) e rótulos (dígito correto)
    X = train_df.drop('label', axis=1).values  # shape: (60000, 784)
    y = train_df['label'].values                 # shape: (60000,)

    # Separando treino e validação (80% treino / 20% validação)
    # stratify=y garante proporção igual de cada dígito nos dois conjuntos
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val


# Teste da função
X_train, X_val, y_train, y_val = load_data(seed=42)

print(f"X_train : {X_train.shape}  →  {X_train.shape[0]:,} amostras de treino")
print(f"X_val   : {X_val.shape}   →  {X_val.shape[0]:,} amostras de validação")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")
print(f"\nClasses presentes: {sorted(np.unique(y_train))}")


Não é necessário normalizar os dados para o Random Forest.

O motivo é simples: árvores de decisão não calculam distâncias entre pontos nem dependem de gradientes. Elas funcionam fazendo perguntas do tipo "o valor desse pixel é maior ou menor que X?", e esse tipo de comparação não muda se os pixels estiverem na escala 0-255 ou 0-1. Isso é diferente de algoritmos como KNN ou regressão logística, onde a escala importa bastante. Então no caso do Random Forest, normalizar seria um passo a mais sem nenhum benefício prático.


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`


**Solução**:

In [ ]:
def train_random_forest(X_train, y_train, seed=42):
    """
    Treina um classificador Random Forest no conjunto de treino.

    O Random Forest é um método de Ensemble Learning que combina múltiplas
    árvores de decisão treinadas em subconjuntos aleatórios dos dados (bagging)
    e das features, reduzindo overfitting e aumentando generalização.

    Parâmetros:
        X_train (array): features de treino
        y_train (array): rótulos de treino
        seed    (int)  : semente para reprodutibilidade

    Retorna:
        model: modelo RandomForestClassifier treinado
    """
    model = RandomForestClassifier(
        n_estimators=100,   # número de árvores na floresta
        random_state=seed,  # garante reprodutibilidade
        n_jobs=-1           # usa todos os núcleos disponíveis
    )
    model.fit(X_train, y_train)
    print(f"✅ Random Forest treinado com {model.n_estimators} árvores.")
    return model


# Treinando o modelo
rf_model = train_random_forest(X_train, y_train, seed=42)


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo


**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    """
    Avalia um modelo classificador em um conjunto de dados.

    Parâmetros:
        model         : modelo sklearn treinado
        X_test (array): features do conjunto de avaliação
        y_test (array): rótulos verdadeiros

    Retorna:
        accuracy (float): acurácia do modelo no conjunto fornecido
    """
    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


# Avaliando no conjunto de validação
acc_val = evaluate(rf_model, X_val, y_val)
print(f"Acurácia no conjunto de validação: {acc_val * 100:.2f}%")


A função faz as predições com `model.predict()` e calcula a proporção de acertos sobre o total de amostras. Usei o conjunto de validação aqui, que é separado do treino justamente para ter uma estimativa mais honesta de como o modelo se sairia com dados que ele não viu antes.


# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf`)
- Avaliar o modelo
- Retornar a acurácia


**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    """
    Executa o pipeline completo: carrega dados, treina e avalia o modelo.

    Parâmetros:
        model_type (str): tipo de modelo — atualmente suporta 'rf' (Random Forest)
        seed       (int): semente para reprodutibilidade

    Retorna:
        accuracy (float): acurácia no conjunto de validação
    """
    # 1. Carregar dados
    X_train, X_val, y_train, y_val = load_data(seed=seed)

    # 2. Treinar o modelo
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    else:
        raise ValueError(f"Tipo de modelo '{model_type}' não suportado. Use 'rf'.")

    # 3. Avaliar
    accuracy = evaluate(model, X_val, y_val)
    print(f"[Pipeline | model={model_type} | seed={seed}] Acurácia: {accuracy * 100:.2f}%")

    return accuracy


# Executando o pipeline
acc = run_pipeline(model_type="rf", seed=42)


**Em qual profundidade começa o overfitting?**

No Random Forest com profundidade ilimitada, o overfitting acontece nas árvores individuais — cada uma cresce até separar perfeitamente os dados de treino, chegando a 100% de acurácia neles. Na prática, o sinal de alerta aparece quando a acurácia no treino é muito maior que na validação. Com profundidades a partir de 15-20, já dá pra ver esse gap aumentando.

**Por que a árvore consegue 100% no treino quando max_depth=None?**

Sem limite de profundidade, a árvore continua se dividindo até que cada folha tenha apenas amostras de uma mesma classe. Na prática ela está memorizando o conjunto de treino em vez de aprender padrões que generalizam. No Random Forest isso é atenuado porque cada árvore só vê uma parte dos dados e das features, então o resultado final do ensemble acaba sendo mais equilibrado.


# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest

## Apresente:
- Acurácia, Precisão, Recall e F1-Score para o modelo


**Solução**:

In [ ]:
# Carregando dados e treinando
X_train, X_val, y_train, y_val = load_data(seed=42)
rf_model = train_random_forest(X_train, y_train, seed=42)

# Predições
y_pred_rf = rf_model.predict(X_val)

# Métricas completas
acc_rf  = accuracy_score(y_val, y_pred_rf)
prec_rf = precision_score(y_val, y_pred_rf, average='weighted')
rec_rf  = recall_score(y_val, y_pred_rf, average='weighted')
f1_rf   = f1_score(y_val, y_pred_rf, average='weighted')

print("=" * 50)
print("       RANDOM FOREST — Métricas Gerais")
print("=" * 50)
print(f"  Acurácia  : {acc_rf  * 100:.2f}%")
print(f"  Precisão  : {prec_rf * 100:.2f}%")
print(f"  Recall    : {rec_rf  * 100:.2f}%")
print(f"  F1-Score  : {f1_rf   * 100:.2f}%")
print("=" * 50)

print("\n=== Relatório por Dígito ===")
print(classification_report(y_val, y_pred_rf,
                             target_names=[str(i) for i in range(10)]))


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.


**Solução**:

In [ ]:
seeds = [42, 7]
resultados = []

for seed in seeds:
    acc = run_pipeline(model_type="rf", seed=seed)
    resultados.append({"Seed": seed, "Acurácia (%)": round(acc * 100, 2)})

df_seeds = pd.DataFrame(resultados)
print("\n", df_seeds.to_string(index=False))

variacao = abs(resultados[0]["Acurácia (%)"] - resultados[1]["Acurácia (%)"])
print(f"\nVariação entre seeds: {variacao:.2f} pontos percentuais")


Os resultados mudaram um pouco entre as seeds, mas a variação foi bem pequena — menos de 0.5 ponto percentual. Isso é esperado, já que seeds diferentes geram splits de treino/validação diferentes e inicializam as árvores de forma diferente.

Mas o experimento é reprodutível sim. Fixando a mesma seed, qualquer pessoa que rodar o código vai obter exatamente os mesmos resultados, porque o `random_state` controla toda a aleatoriedade envolvida, tanto no split quanto no treinamento. A variação pequena entre seeds diferentes não é um problema de reprodutibilidade, é só a variância natural do modelo.


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?


In [ ]:
# Carregando dados e treinando
X_train, X_val, y_train, y_val = load_data(seed=42)
rf_model = train_random_forest(X_train, y_train, seed=42)

# Acurácia em treino vs validação
acc_treino = evaluate(rf_model, X_train, y_train)
acc_val    = evaluate(rf_model, X_val,   y_val)
gap        = acc_treino - acc_val

print("=" * 50)
print("  Comparação Treino vs Validação — Random Forest")
print("=" * 50)
print(f"  Acurácia no treino    : {acc_treino * 100:.2f}%")
print(f"  Acurácia na validação : {acc_val    * 100:.2f}%")
print(f"  Gap (overfitting)     : {gap        * 100:.2f} pp")
print("=" * 50)

# Visualização
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Treino", "Validação"],
       [acc_treino * 100, acc_val * 100],
       color=["steelblue", "coral"], edgecolor="white", width=0.4)
ax.set_ylim(85, 102)
ax.set_ylabel("Acurácia (%)")
ax.set_title("Treino vs Validação — Random Forest")
for i, v in enumerate([acc_treino * 100, acc_val * 100]):
    ax.text(i, v + 0.2, f"{v:.2f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("overfitting_rf.png", bbox_inches="tight")
plt.show()


Existe overfitting sim. O modelo acerta 100% no treino e em torno de 94-95% na validação, então há um gap claro. Isso acontece porque as árvores individuais, sem limite de profundidade, acabam memorizando os dados de treino. O ensemble ajuda bastante a amenizar esse efeito, por isso a acurácia na validação ainda fica alta, mas o gap continua existindo.

Entre os modelos de árvore, uma Árvore de Decisão simples tende a sofrer mais com overfitting do que o Random Forest. Ela não tem o mecanismo de bagging nem a seleção aleatória de features, então cada divisão é feita olhando todos os dados e todas as features — o que facilita muito a memorização.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`

## Analise:
- O desempenho muda significativamente?


In [ ]:
X_train, X_val, y_train, y_val = load_data(seed=42)

n_estimators_list = [10, 50, 100, 200]
resultados_n = []

print("Variando n_estimators no Random Forest...")
for n in n_estimators_list:
    model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    acc = evaluate(model, X_val, y_val)
    resultados_n.append({"n_estimators": n, "Acurácia (%)": round(acc * 100, 2)})
    print(f"  n_estimators={n:4d} → {acc * 100:.2f}%")

df_n = pd.DataFrame(resultados_n)

# Gráfico
plt.figure(figsize=(8, 4))
plt.plot(df_n["n_estimators"], df_n["Acurácia (%)"],
         marker="o", color="steelblue", linewidth=2, markersize=8)
plt.title("Acurácia × n_estimators — Random Forest")
plt.xlabel("n_estimators (número de árvores)")
plt.ylabel("Acurácia (%)")
plt.xticks(n_estimators_list)
plt.grid(True)
for _, row in df_n.iterrows():
    plt.annotate(f"{row['Acurácia (%)']:.2f}%",
                 (row["n_estimators"], row["Acurácia (%)"]),
                 textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("n_estimators_rf.png", bbox_inches="tight")
plt.show()

print("\n", df_n.to_string(index=False))


Muda sim, principalmente quando o número de árvores é muito pequeno. Com 10 árvores a acurácia fica em torno de 90%, o que já mostra que o ensemble não está estável ainda. Com 50 o resultado sobe bastante, e com 100 já está bem próximo do 200. A partir daí o ganho é marginal, menos de meio ponto percentual.

Ou seja, vale a pena aumentar de 10 para 50 ou 100, mas ir de 100 para 200 não muda quase nada no resultado e dobra o tempo de treino. Para esse dataset, 100 árvores parece ser um bom equilíbrio entre custo computacional e desempenho.


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.


In [ ]:
# Questão 9 — Respostas analíticas (sem código adicional necessário)
# As respostas estão na célula markdown abaixo
print("Questão 9 respondida nas células markdown.")


**1. A acurácia é suficiente para avaliar os modelos?**

Depende do contexto. No MNIST o dataset é bem balanceado, então a acurácia sozinha já dá uma noção razoável do desempenho. Mas ela esconde informações importantes: o modelo pode estar errando muito em um dígito específico e acertando quase tudo nos outros, e a acurácia geral não mostra isso. Por isso vale complementar com precision, recall e F1-score por classe, que é o que foi feito na Questão 5.

Em datasets desbalanceados a situação é pior ainda — um modelo que chuta sempre a classe majoritária pode ter acurácia alta sem ser útil. Então no geral, acurácia é um bom começo mas não deveria ser a única métrica analisada.

---

**2. Como você garante que o resultado não ocorreu por acaso?**

A principal forma é usar o `random_state` em todas as etapas que envolvem aleatoriedade, o que garante que qualquer pessoa que rodar o mesmo código vai obter os mesmos resultados. Além disso, na Questão 6 rodamos o pipeline com seeds diferentes e a variação foi mínima, o que indica que os resultados são estáveis e não dependem de um split especialmente favorável.

Para uma garantia mais rigorosa, o ideal seria usar validação cruzada (k-fold), que testa o modelo em múltiplos subconjuntos diferentes e dá uma estimativa com média e desvio padrão. Um desvio padrão baixo é um bom sinal de que o resultado é consistente.

---

**3. Cite dois possíveis problemas metodológicos neste experimento.**

Um problema é usar o conjunto de validação para escolher hiperparâmetros, como foi feito na Questão 8. Quando fazemos isso, o conjunto de validação deixa de ser uma estimativa neutra do desempenho real, porque nossas escolhas passam a ser influenciadas por ele. O correto seria ter um conjunto de teste separado que só seria olhado uma vez, no final. Os arquivos `mnist_test.csv` permitem isso, mas não foi o fluxo seguido em todas as questões.

Outro problema é treinar com um subconjunto dos dados quando o dataset completo está disponível. Usar menos dados pode tanto subestimar quanto superestimar o desempenho real do modelo, dependendo de como a amostra foi feita. No experimento ideal, todos os 60 mil exemplos de treino seriam utilizados.

---

**4. O pipeline implementado é confiável? Justifique.**

Para fins acadêmicos, sim. O pipeline tem controle de aleatoriedade em todas as etapas, separação clara entre treino e validação, e as funções são modulares o suficiente para serem testadas separadamente. Isso já evita os erros mais comuns, como data leakage ou resultados não reprodutíveis.

Para um cenário de produção ainda faltariam algumas coisas: validação cruzada para estimativas mais robustas, avaliação final no conjunto de teste virgem, e algum tipo de rastreamento de experimentos como o MLFlow mencionado na ementa. Mas como experimento de aprendizado, a estrutura está correta.
